In [2]:
from datasets import load_dataset
from transformers import pipeline
import re

classifier = pipeline("sentiment-analysis", model="lvwerra/distilbert-imdb")
tokenizer = classifier.tokenizer

dataset = load_dataset("stanfordnlp/imdb", split="test")
sample = dataset.shuffle(seed=42).select(range(500))
texts = list(sample["text"])
labels = list(sample["label"])

Device set to use mps:0


In [3]:
def clean(text):
    return re.sub(r"<br\s*/?>", " ", text)

def head_tail(text, tokenizer, max_len=510, head=128):
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_len:
        return text
    ids = ids[:head] + ids[-(max_len - head):]
    return tokenizer.decode(ids)

In [4]:
processed = [head_tail(clean(t), tokenizer) for t in texts]
results = classifier(processed, truncation=True, batch_size=16)

label_map = {"NEGATIVE": 0, "POSITIVE": 1}
preds = [label_map[r["label"]] for r in results]

Token indices sequence length is longer than the specified maximum sequence length for this model (590 > 512). Running this sequence through the model will result in indexing errors


In [5]:
from sklearn.metrics import classification_report, confusion_matrix

correct = sum(p == l for p, l in zip(preds, labels))
print(f"got {correct} right in {len(labels)}, Accuracy: {correct / len(labels):.4f}")
print(classification_report(labels, preds, target_names=["negative", "positive"]))
print(confusion_matrix(labels, preds))

got 460 right in 500, Accuracy: 0.9200
              precision    recall  f1-score   support

    negative       0.92      0.92      0.92       254
    positive       0.92      0.92      0.92       246

    accuracy                           0.92       500
   macro avg       0.92      0.92      0.92       500
weighted avg       0.92      0.92      0.92       500

[[233  21]
 [ 19 227]]


In [6]:
errors = [(t, y, p) for t, y, p in zip(texts, labels, preds) if y != p]
for t, y, p in errors[:5]:
    print(f"label {y} / pred {p}: {t[:300]}...\n")

label 0 / pred 1: A holiday on a boat, a married couple, an angry waiter and a shipwreck is the reason to this films beginning.<br /><br />I like boobs. No question about that. But when the main character allies with whoever happens to have the most fish at the moment, mostly by having sex with them and playing the r...

label 0 / pred 1: Coming from Kiarostami, this art-house visual and sound exposition is a surprise. For a director known for his narratives and keen observation of humans, especially children, this excursion into minimalist cinematography begs for questions: Why did he do it? Was it to keep him busy during a vacation...

label 0 / pred 1: In the early 00's, production companies had a short-lived craze for supernatural genre movies in France after "The Crimson Rivers" and "Brotherhood of the Wolf" turned out to be hits, so several movies were green-lit or saved from their "direct-to-video" fate. However, France, as opposed to the US, ...

label 1 / pred 0: Sex, drugs, r